Steps to preprocess MODIS data for phenology tracking. These methods are 100% adapted from Jerod Merkle and Shannon Albeke's workflow to process data in R. 

Steps 1a, 1b and the first function in step 2 (ndviCALC_clean) all run within GEE (~10 seconds). However, GEE runs into memory issues when trying to fit the curves on the image collections (the spatial data that is stored in GEE). So, we export the image collections as rasters and then will read it back in.

Conversion to Python and GEE by Tayler LaSharr

October 2024

In [22]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import numpy as np
from scipy.optimize import curve_fit # for fitting the DLC curves (similar to NLS in R)
import multiprocessing as mp


## if you need to authenticate GEE on your machine, use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module (change 'tlasharr' to your project name)
ee.Initialize(project='ee-tlasharr')

Step 1a. Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [23]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_winter_month_day = '11-01'
end_winter_month_day = '03-01'

## get the time period we care about
start_date_data = '2001-01-01'
end_date_data = '2012-12-31'

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])

## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )
modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))


Step 1b. Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function: removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

Second function: creates a Snow Free Mask. This function will create an image collection called 'SnowCover' that has an image with pixels set to 1 (snow) or 0 (no snow) for each image during the time period you are interested in. This is the equivalent of the 'SnowCover' rasters produced in step 1b of the R code.

In [24]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information from the State band of the MODIS data
def extract_snow(image):
    
    ## get the state band
    state = image.select('State')

    ## bit 12 represents snow/ice
    snow_mod35 = state.bitwiseAnd(1 << 12).neq(0)  # 1 if snow/ice is present, 0 otherwise

    ## add the snow band to the image
    snow_band = snow_mod35.rename('Snow')

    ## carry over the system:time_start property from the original image (this is not saved in the State band, so you need to explicitly call it back in)
    return snow_band.set('system:time_start', image.get('system:time_start'))


## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = modis_ndvi.map(extract_snow)


Step 2. Calculations - this code is broken up by the 3 main functions used in the R code (ndviCALC_clean, ndviCALC_dlc, and ndviCALC_vals). Additionally, it exports the image collections at the end of the first function and reads them back in as rasters.

ndviCALC_clean:

Perform all steps in the ndviCALC_clean function on the filtered MODIS data. These functions duplicates the steps in the ndviCALC_clean function that occurs during step 2 in the R code. I seperated these out into substeps because it was failing when I tried running it in a single function. This could likely be optimized more, but it still runs incredibly efficiently as written (i.e., a few seconds for the state of WY).

In [32]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):

    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.gte(0).And(image.lte(2)))

    NDVICollection = NDVICollection.map(reclassify_outliers)
    
    return NDVICollection

## step 2: reclassify values during winter as 2
def ndviCALC_step2(NDVICollection, start_winter_month_day='11-01', end_winter_month_day='03-30'):
    
    ## split the winter start and end dates into months and days
    start_winter_month, start_winter_day = map(int, start_winter_month_day.split('-'))
    end_winter_month, end_winter_day = map(int, end_winter_month_day.split('-'))

    ## function to classify whether the date is in the winter period
    def classify_winter(image):
        
        ## extract the date of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_winter_month, start_winter_day)
        winter_end = ee.Date.fromYMD(year, end_winter_month, end_winter_day)

        ## use date difference to identify if the image falls within winter
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## mask the image during winter (i.e., replace NDVI values with 2 during winter)
        image_winter_adjusted = image.where(is_in_winter, ee.Image.constant(2))

        return image_winter_adjusted

    ## map the function over the collection
    NDVICollection_winter_adjusted = NDVICollection.map(classify_winter)
    
    return NDVICollection_winter_adjusted


## step 3: check if pixel has been snow-free for 3 consecutive images
def ndviCALC_step3(NDVICollection, SnowCollection):
    
    def snow_free_for_3(image, prev_image1, prev_image2):
        ## get current and previous snow cover (unmask to avoid gaps)
        current_snow = image.unmask(1)
        prev_snow1 = prev_image1.unmask(1)
        prev_snow2 = prev_image2.unmask(1)

        ## check if the pixel is snow-free for 3 consecutive images (snow-free if sum == 0)
        snow_free_3_consec = current_snow.add(prev_snow1).add(prev_snow2).eq(0)

        return snow_free_3_consec
    
    ## convert both the NDVI and Snow collections to lists of images
    ndvi_images = NDVICollection.toList(NDVICollection.size())
    snow_images = SnowCollection.toList(SnowCollection.size())

    ## initialize the collection with the first two images (can't apply the snow-free logic to these)
    snow_free_collection = ee.ImageCollection([ndvi_images.get(0), ndvi_images.get(1)])

    ## loop through the rest of the images (starting from the third)
    for i in range(2, NDVICollection.size().getInfo()):
        current_image_ndvi = ee.Image(ndvi_images.get(i))
        current_image_snow = ee.Image(snow_images.get(i))
        prev_image1_snow = ee.Image(snow_images.get(i - 1))
        prev_image2_snow = ee.Image(snow_images.get(i - 2))

        ## apply the snow-free logic to the snow images
        snow_free_image = snow_free_for_3(current_image_snow, prev_image1_snow, prev_image2_snow)

        ## mask the NDVI image based on the snow-free condition
        ndvi_snow_free_masked = current_image_ndvi.updateMask(snow_free_image)

        ## append the result to the collection
        snow_free_collection = snow_free_collection.merge(ee.ImageCollection([ndvi_snow_free_masked]))

    return snow_free_collection



## step 4: mask pixels that are in winter or covered by snow
def ndviCALC_step4(NDVICollection, SnowCollection):

    def mask_winter_snow(ndvi_image):

        ## get the corresponding snow image for the same date
        snow_image = SnowCollection.filterDate(ndvi_image.date(), ndvi_image.date()).first()

        ## ensure that snow_image exists, if not, create an empty image with value 0
        snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))

        ## mask pixels where snow cover is equal to 2 (as set in earlier steps) or during winter
        masked_image = ndvi_image.updateMask(snow_image.neq(2))  # mask where Snow equals 2 (snow-covered)

        return masked_image

    ## apply the snow and winter mask to the NDVIFiltered collection
    NDVIFiltered_masked = NDVICollection.map(mask_winter_snow)

    return NDVIFiltered_masked


## step 5: apply median filter
def ndviCALC_step5(image_collection, window_size=3):
    
    ## ensure that the NDVI is in float format for precision
    image_collection = image_collection.map(lambda img: img.select('NDVI').toFloat()) 

    def smooth_image(current_image):
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        image_index = all_images_list.indexOf(current_image)
        half_window = ee.Number(window_size // 2)

        ## calculate the start and end indices for the temporal window
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## get the window of images and create a new image collection
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median NDVI for the current window
        median_image = window_collection.median().toFloat()

        ## copy the properties of the current image and return the median NDVI
        return median_image.copyProperties(current_image, current_image.propertyNames())

    # Apply the temporal smoothing function to the entire collection
    smoothed_collection = image_collection.map(smooth_image)
    
    return smoothed_collection


## step 6: calculate the floor (0.025 quantile)
def ndviCALC_step6(NDVICollection, quantile_value=2.5):

    def floor_quantile(ndvi_image):
        
        ## calculate the 0.025 quantile for each image
        quantile = ndvi_image.reduceRegion(
            reducer=ee.Reducer.percentile([quantile_value]),
            geometry=ndvi_image.geometry(),
            scale=250,
            maxPixels=1e9
        )
        
        ## extract the quantile value (default to 0 if not found)
        qt_value = ee.Number(quantile.get('NDVI', 0))
        
        ## floor NDVI values below the quantile value
        floored_image = ndvi_image.where(ndvi_image.lt(qt_value), qt_value)
        
        return floored_image

    ## apply the quantile flooring to the entire collection
    floored_collection = NDVICollection.map(floor_quantile)
    
    return floored_collection

## step 7: floor NDVI values during winter but leave snow-covered pixels as NA
def ndviCALC_step7(NDVICollection, SnowCollection, start_winter_month_day='11-01', end_winter_month_day='03-30', quantile_value=2.5):
    
    ## split the start and end dates into month and day components
    start_winter_month, start_winter_day = map(int, start_winter_month_day.split('-'))
    end_winter_month, end_winter_day = map(int, end_winter_month_day.split('-'))

    def floor_winter_pixels(ndvi_image):
        
        ## extract the date of the NDVI image
        date = ee.Date(ndvi_image.get('system:time_start'))
        year = date.get('year')

        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_winter_month, start_winter_day)
        winter_end = ee.Date.fromYMD(year, end_winter_month, end_winter_day)

        ## check if the image date is within the winter period
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## create a mask for if it's winter or not
        winter_mask = ee.Image.constant(is_in_winter)

        ## calculate the 0.025 quantile for the NDVI image
        quantile = ndvi_image.reduceRegion(
            reducer=ee.Reducer.percentile([quantile_value]),
            geometry=ndvi_image.geometry(),
            scale=250,
            maxPixels=1e9
        ).get('NDVI', 0)  # Default to 0 if the quantile is not found
        
        ## convert the quantile value into an Image
        quantile_image = ee.Image.constant(quantile)

        ## floor NDVI values below the quantile during the winter period
        ndvi_floored_winter = ndvi_image.where(winter_mask.And(ndvi_image.lt(quantile_image)), quantile_image)

        ## retrieve the corresponding snow image for the same date
        snow_image = SnowCollection.filter(ee.Filter.date(ndvi_image.date())).first()

        ## ensure that snow_image exists; if not, create a default image with 0 (no snow)
        snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))

        ## keep snow-covered pixels as masked (NA)
        final_masked_ndvi = ndvi_floored_winter.updateMask(snow_image.neq(1))  # Mask where snow == 1 (snow-covered)

        return final_masked_ndvi

    ## apply the flooring and masking logic to the entire NDVI collection
    floored_winter_collection = NDVICollection.map(floor_winter_pixels)

    return floored_winter_collection


## step 8: scale NDVI values between 0 and 1 using the 97.5% quantile
def ndviCALC_step8(NDVICollection, quantile_value=97.5):

    def scale_ndvi(ndvi_image):
        
        ## calculate the min and 97.5% quantile NDVI values
        min_val = ndvi_image.reduceRegion(
            reducer=ee.Reducer.min(),
            geometry=ndvi_image.geometry(),
            scale=250,
            maxPixels=1e9
        ).get('NDVI')

        max_val = ndvi_image.reduceRegion(
            reducer=ee.Reducer.percentile([quantile_value]),
            geometry=ndvi_image.geometry(),
            scale=250,
            maxPixels=1e9
        ).get('NDVI')

        ## scale NDVI values to [0, 1] based on min and max
        scaled_image = ndvi_image.unitScale(min_val, max_val)
        
        return scaled_image

    ## pply scaling to the NDVI collection
    scaled_collection = NDVICollection.map(scale_ndvi)

    return scaled_collection


## here we could write over the image collections, but it doesn't cause any memory issues and I still am not super confident in GEE stuff and how
## overwriting works for image collections, so am not overwriting them yet.

## run step 1
NDVIFiltered_reclassified = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")

## run step 2
NDVIFiltered_winter_classified = ndviCALC_step2(NDVIFiltered_reclassified, start_winter_month_day='11-01', end_winter_month_day='03-30')
print("Step 2 complete: Reclassified Values during winter to NAs.")

## run step 3
NDVIFiltered_snow_free = ndviCALC_step3(NDVIFiltered_winter_classified, SnowCover)
print("Step 3 complete: Check if pixels have been snow-free for 3 consecutive images.")

## run step 4
NDVIFiltered_masked = ndviCALC_step4(NDVIFiltered_snow_free, SnowCover)
print("Step 4 complete: Mask pixels that are in winter or covered by snow.")

## run step 5
NDVIFiltered_temporal_smoothed = ndviCALC_step5(NDVIFiltered_masked, window_size=3)
print("Step 5 complete: Applied moving window with a window size of 3.")

## run step 6
NDVIFiltered_floored = ndviCALC_step6(NDVIFiltered_temporal_smoothed, quantile_value=2.5)
print("Step 6 complete: Calculate the floor (0.025 quantile).")

## run step 7
NDVIFiltered_winter_floored = ndviCALC_step7(NDVIFiltered_floored, SnowCover, start_winter_month_day='11-01', end_winter_month_day='03-30', quantile_value=2.5)
print("Step 7 complete: Winter pixels floored, snow-covered pixels remain masked.")

## run step 8
NDVIFiltered_scaled = ndviCALC_step8(NDVIFiltered_winter_floored, quantile_value=97.5)
print("Step 8 complete: NDVI values scaled between 0 and 1.")

Step 1 complete: Removed outliers from data.
Step 2 complete: Reclassified Values during winter to NAs.
Step 3 complete: Check if pixels have been snow-free for 3 consecutive images.
Step 4 complete: Mask pixels that are in winter or covered by snow.
Step 5 complete: Applied moving window with a window size of 3.
Step 6 complete: Calculate the floor (0.025 quantile).
Step 7 complete: Winter pixels floored, snow-covered pixels remain masked.
Step 8 complete: NDVI values scaled between 0 and 1.


The next thing we need to do is run the ndviCALC_dlc function and onwards. However, GEE can't handle fitting the curves with the amount of memory in its system, so we are going to export the image collections and read them back in as rasters.

In [33]:
## function to export each image as a layer within a GeoTIFF per year
def export_yearly_ndvi_as_layers(NDVIFiltered_scaled, start_year, end_year, export_folder):
    for year in range(start_year, end_year + 1):
        
        ## filter by year
        yearly_collection = NDVIFiltered_scaled.filterDate(f'{year}-01-01', f'{year}-12-31')
        
        ## convert the image collection to a single image with each band corresponding to an image in the collection
        def band_names(image, prev_image):
            prev_image = ee.Image(prev_image)
            updated = prev_image.addBands(image.select('NDVI'))
            return updated

        ## create an initial image with a dummy band to start adding bands
        dummy_image = yearly_collection.first().select('NDVI').multiply(0).rename('dummy')
        
        ## accumulate all images as bands
        yearly_image = ee.Image(yearly_collection.iterate(band_names, dummy_image))
        
        ## remove the initial placeholder dummy band
        band_list = yearly_image.bandNames().remove('dummy')
        yearly_image = yearly_image.select(band_list)
        
        ## define export task
        export_task = ee.batch.Export.image.toDrive(
            image=yearly_image,
            description=f'NDVI_{year}',
            folder=export_folder,
            fileNamePrefix=f'NDVI_{year}',
            scale=30, # Adjust scale as needed
            region=yearly_image.geometry(),
            maxPixels=1e13
        )
        
        ## start the export task
        export_task.start()

export_yearly_ndvi_as_layers(NDVIFiltered_scaled, 2001, 2002, 'NDVI_Exports')

ndviCALC_dlc: Calculates the double logisitic curve parameters and returns and image colleciton. The sequence here also differs slightly from the R function, and I define a few function outside of the larger function first.

In [5]:
## the dlc model used in the function
def dlc_model(time, xmidS, scalS, xmidA, scalA):
    return (1 / (1 + np.exp((xmidS - time) / scalS))) - (1 / (1 + np.exp((xmidA - time) / scalA)))

## subset to get the yearly NDVI data 
def get_yearly_ndvi(cleanVals, start_date, end_date):
    yearly_data = cleanVals.filterDate(start_date, end_date)
    
    def extract_ndvi(image):
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
        ndvi = image.reduceRegion(ee.Reducer.mean(), geometry=image.geometry(), scale=250).get('NDVI')
        return ee.Feature(None, {'date': date, 'NDVI': ndvi})
    
    return yearly_data.map(extract_ndvi).getInfo()

## fit the DLC curves to the data
def fit_dlc(cleanVals, start_date, end_date):
    
    ## get an empty object to store the dlc params in, start year, end year, and range
    dlc_params = []
    start_year = int(start_date.split('-')[0])
    end_year = int(end_date.split('-')[0])
    years = range(start_year, end_year + 1)

    ## for each year, fit the curves
    for year in years:

        ## tell you what year you're processing
        print(f"Processing year: {year}")

        ## get the start and end date and then all the data for the year
        year_start = f'{year}-01-01'
        year_end = f'{year}-12-31'
        year_data = get_yearly_ndvi(cleanVals, year_start, year_end)
        
        ## get the dates from the subset NDVI data (dates are an attribute of the image collection)
        dates = [d['properties']['date'] for d in year_data]

        ## get the NDVI values from the subset NDVI da
        ndvi_values = [d['properties']['NDVI'] for d in year_data]

        ## convert time and ndvi to a pandas dataframe
        time = pd.to_datetime(dates).dayofyear.to_numpy()
        ndvi_values = np.array(ndvi_values)
        
        ## if there were less than 4 pixels with NDVi values in the entire sequence, NA all pizels in the time series
        if len(ndvi_values) < 4:
            dlc_params.append([np.nan] * 8)
            continue

        try:
            ## fit the curves, the bounds are the lower and upper limits
            popt, _ = curve_fit(dlc_model, time, ndvi_values, bounds=([30, 1, 167, 1], [228, 30, 288, 35]))
            dlc_params.append(popt.tolist())

        except Exception as e:
            ## print if there was an error
            print(f"Error fitting year {year}: {e}")
            dlc_params.append([np.nan] * 8)
    
    return dlc_params

In [7]:
## we were running into memory/computational limitations, so here we can split the bounding box up into smaller areas and process them in batches

## function to seperate out the bounding boxes
def divide_bounding_box(bounding_box, num_divisions):
    bounds = bounding_box.bounds().getInfo()['coordinates'][0]
    lon_min, lat_min, lon_max, lat_max = np.min(bounds, axis=0)[0], np.min(bounds, axis=0)[1], np.max(bounds, axis=0)[0], np.max(bounds, axis=0)[1]
    
    lon_range = np.linspace(lon_min, lon_max, num_divisions + 1)
    lat_range = np.linspace(lat_min, lat_max, num_divisions + 1)
    
    boxes = []
    for i in range(num_divisions):
        for j in range(num_divisions):
            box = ee.Geometry.Rectangle([lon_range[i], lat_range[j], lon_range[i+1], lat_range[j+1]])
            boxes.append(box)
    
    return boxes

## function to fit the DLC curves in batches
def fit_dlc_batch(cleanVals, start_date, end_date, bounding_box, num_divisions=2):
    dlc_params = []
    boxes = divide_bounding_box(bounding_box, num_divisions)
    total_boxes = len(boxes)
    for idx, box in enumerate(boxes):
        print(f"Processing box {idx + 1} out of {total_boxes}: {box.getInfo()}")
        cleanVals_clipped = cleanVals.map(lambda img: img.clip(box))
        
        params = fit_dlc(cleanVals_clipped, start_date, end_date)
        dlc_params.extend(params)
    
    return dlc_params


dlc_params = fit_dlc_batch(NDVIFiltered_scaled, start_date_data, end_date_data, bounding_box, num_divisions=10)
print(dlc_params)


Processing box 1 out of 100: {'type': 'Polygon', 'coordinates': [[[-110.80210107405063, 41.723905062187754], [-110.58709502524755, 41.723905062187754], [-110.58709502524755, 41.89258463143338], [-110.80210107405063, 41.89258463143338], [-110.80210107405063, 41.723905062187754]]]}
Processing year: 2001


EEException: Computation timed out.